In [5]:
import h5py
import numpy as np
import os

def write_hdf5_file(quasar_list, file_path):
    print(f"Writing {len(quasar_list)} quasars to {file_path}", flush=True)
    with h5py.File(file_path, "w") as hdf:  # overwrite mode
        for quasar in quasar_list.values():
            object_id = quasar["object_id"]
            group = hdf.create_group(object_id)
            for key, value in quasar.items():
                if key == "object_id":
                    continue
                if isinstance(value, dict):
                    sub_group = group.create_group(key)
                    for sub_key, sub_value in value.items():
                        sub_group.create_dataset(
                            sub_key,
                            data=sub_value,
                            compression='gzip',
                            compression_opts=9,
                            fletcher32=True
                        )
                elif isinstance(value, (int, float, str, bytes)) or (
                    hasattr(value, 'shape') and hasattr(value, 'dtype')
                ):
                    group.create_dataset(
                        key,
                        data=value,
                        compression='gzip',
                        compression_opts=9,
                        fletcher32=True
                    )
                else:
                    try:
                        group.create_dataset(
                            key,
                            data=value,
                            compression='gzip',
                            compression_opts=9,
                            fletcher32=True
                        )
                    except TypeError:
                        group.attrs[key] = str(value)


def merge_hdf5_files(file_list, output_file):
    quasar_list = {}
    for file_path in file_list:
        with h5py.File(file_path, "r") as hdf:
            for obj_id in hdf.keys():
                quasar = {"object_id": obj_id}
                group = hdf[obj_id]
                for key in group.keys():
                    if isinstance(group[key], h5py.Group):
                        quasar[key] = {sub_key: group[key][sub_key][()] for sub_key in group[key].keys()}
                    else:
                        quasar[key] = group[key][()]
                quasar_list[obj_id] = quasar
    write_hdf5_file(quasar_list, output_file)


merge_hdf5_files(
    file_list=[
        "data/may4_objs_tauwavelength.h5",
        "data/may4_objs_tauwavelength_2.h5",
        "data/may4_objs_tauwavelength_3.h5",
    ],
    output_file="data/may4_objs_tauwavelength_merged.h5"
)

Writing 1906 quasars to data/may4_objs_tauwavelength_merged.h5


In [9]:
def load_hdf5_file(file_path):
    quasar_list = []
    file = h5py.File(file_path, "r")
    for obj_id in file.keys():
        quasar = {"object_id": obj_id}
        group = file[obj_id]
        for key in group.keys():
            if isinstance(group[key], h5py.Group):
                quasar[key] = {sub_key: group[key][sub_key][()] for sub_key in group[key].keys()}
            else:
                quasar[key] = group[key][()]
        quasar_list.append(quasar)
    return quasar_list

load_hdf5_file("data/may4_objs_tauwavelength_merged.h5");